# Padding Parameter Benchmark

This notebook benchmarks site-level modification detection across different
DTW **padding** values (0, 1, 2) using the default 3-state unsupervised
kNN→HMM pipeline.

Padding controls how many flanking positions' signals are concatenated
before computing pairwise DTW distances:

| Padding | Signal window | Description |
|---------|---------------|-------------|
| 0 | 1 position | Single 5-mer signal (default) |
| 1 | 3 positions | ±1 flanking positions concatenated |
| 2 | 5 positions | ±2 flanking positions concatenated |

### Metrics compared

1. **Read-level**: AUROC, AUPRC, F1 (per-read P(mod) vs ground truth)
2. **Site-level**: significant sites (padj < 0.05), mod_ratio accuracy
3. **Resource usage**: wall time (f5c, DTW, HMM), peak memory

## 0. Setup & Paths

In [ ]:
import gc
import logging
import os
import resource
import shutil
import sys
import tempfile
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
warnings.filterwarnings("ignore", category=RuntimeWarning)

TESTDATA = Path("../testdata").resolve()
assert TESTDATA.exists(), f"testdata directory not found: {TESTDATA}"

NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"
IVT_BAM      = TESTDATA / "control_0.bam"
IVT_FASTQ    = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5    = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"
REF_FASTA    = TESTDATA / "ref.fa"

for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "\u2705" if path.exists() else "\u274c MISSING"
    print(f"  {status}  {label:15s}  {path}")

OUTPUT_DIR = Path("../output/padding_benchmark").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

PADDING_VALUES = [0, 1, 2]

## 1. Known Modifications

In [ ]:
POSITION_OFFSET = 3

KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Psi", "pseudouridine"),
    ("ecoli23S", 746):  ("Psi", "pseudouridine"),
    ("ecoli23S", 955):  ("Psi", "pseudouridine"),
    ("ecoli23S", 1911): ("Psi", "pseudouridine"),
    ("ecoli23S", 1917): ("Psi", "pseudouridine"),
    ("ecoli23S", 2457): ("Psi", "pseudouridine"),
    ("ecoli23S", 2504): ("Psi", "pseudouridine"),
    ("ecoli23S", 2580): ("Psi", "pseudouridine"),
    ("ecoli23S", 2604): ("Psi", "pseudouridine"),
    ("ecoli23S", 2605): ("Psi", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m66A) ---
    ("ecoli16S", 1518): ("m66A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m66A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Single-occurrence modifications ---
    ("ecoli23S", 2498): ("Cm",    "2\u2032-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D",     "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm",    "2\u2032-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um",    "2\u2032-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C",  "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G",   "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A",   "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Psi", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U",   "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm",  "N4,2\u2032-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_PIPELINE_SET = set(KNOWN_MOD_PIPELINE.keys())

print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")

## 2. Shared Pre-processing (f5c eventalign)

f5c eventalign is run **once** — the signal data is the same regardless of
padding. Only the DTW signal extraction and pairwise computation differ.

In [ ]:
from baleen.eventalign._bam import (
    get_contig_stats, filter_contigs, validate_bam, split_bam_contig,
)
from baleen.eventalign._f5c import check_f5c, index_fastq_blow5, index_blow5, run_eventalign
from baleen.eventalign._signal import (
    group_signals_by_position,
    get_common_positions,
    extract_signals_for_dtw,
    extract_signals_for_dtw_padded,
)
from baleen.eventalign._pipeline import PositionResult, ContigResult
from baleen._cuda_dtw import dtw_pairwise_varlen, CUDA_AVAILABLE

MIN_DEPTH = 15

f5c_version = check_f5c()
print(f"f5c version: {f5c_version}")
print(f"CUDA available: {CUDA_AVAILABLE}")

print("Indexing...")
index_fastq_blow5(NATIVE_FASTQ, NATIVE_BLOW5)
index_fastq_blow5(IVT_FASTQ, IVT_BLOW5)
index_blow5(NATIVE_BLOW5)
index_blow5(IVT_BLOW5)

validate_bam(NATIVE_BAM)
validate_bam(IVT_BAM)
native_stats = get_contig_stats(NATIVE_BAM)
ivt_stats = get_contig_stats(IVT_BAM)
passed_contigs, _ = filter_contigs(native_stats, ivt_stats, min_depth=float(MIN_DEPTH))
print(f"Contigs passing filter: {passed_contigs}")

In [ ]:
# Run f5c eventalign ONCE and cache parsed signals per contig
eventalign_signals: dict[str, tuple[dict, dict, list]] = {}  # contig -> (native_by_pos, ivt_by_pos, common)

tmp_root = Path(tempfile.mkdtemp(prefix="baleen-padding-bench-"))
print(f"Temp dir: {tmp_root}")

t0_f5c = time.time()
try:
    for contig in passed_contigs:
        print(f"\nProcessing contig: {contig}")
        contig_tmp = tmp_root / contig
        contig_tmp.mkdir(parents=True, exist_ok=True)

        native_contig_bam = split_bam_contig(NATIVE_BAM, contig, contig_tmp / "native")
        ivt_contig_bam = split_bam_contig(IVT_BAM, contig, contig_tmp / "ivt")

        native_tsv = contig_tmp / "native.eventalign.tsv"
        ivt_tsv = contig_tmp / "ivt.eventalign.tsv"

        print("  Running f5c eventalign (native)...")
        run_eventalign(native_contig_bam, REF_FASTA, NATIVE_FASTQ, NATIVE_BLOW5, native_tsv, rna=True)
        print("  Running f5c eventalign (IVT)...")
        run_eventalign(ivt_contig_bam, REF_FASTA, IVT_FASTQ, IVT_BLOW5, ivt_tsv, rna=True)

        native_by_pos = group_signals_by_position(native_tsv)
        ivt_by_pos = group_signals_by_position(ivt_tsv)
        common = get_common_positions(native_by_pos, ivt_by_pos)

        eventalign_signals[contig] = (native_by_pos, ivt_by_pos, common)
        print(f"  \u2192 {len(common)} common positions")

finally:
    shutil.rmtree(tmp_root, ignore_errors=True)
    print(f"\nCleaned up temp dir: {tmp_root}")

f5c_time = time.time() - t0_f5c
print(f"\nf5c eventalign total: {f5c_time:.1f}s (shared across all padding values)")

## 3. Run DTW + HMM + Aggregation for Each Padding Value

For each padding value, compute pairwise DTW distances, run the full
3-state unsupervised kNN→HMM pipeline, then aggregate to site-level results.

In [ ]:
from baleen.eventalign import (
    compute_sequential_modification_probabilities,
    aggregate_all,
    write_site_tsv,
)


def compute_dtw_for_padding(
    padding: int,
    eventalign_signals: dict[str, tuple[dict, dict, list]],
) -> dict[str, ContigResult]:
    """Compute pairwise DTW distances for a given padding value."""
    results: dict[str, ContigResult] = {}

    for contig, (native_by_pos, ivt_by_pos, common) in eventalign_signals.items():
        position_results: dict[int, PositionResult] = {}

        for pos in common:
            if padding > 0:
                native_names, native_sigs = extract_signals_for_dtw_padded(
                    native_by_pos, pos, padding,
                )
                ivt_names, ivt_sigs = extract_signals_for_dtw_padded(
                    ivt_by_pos, pos, padding,
                )
            else:
                native_names, native_sigs = extract_signals_for_dtw(native_by_pos[pos])
                ivt_names, ivt_sigs = extract_signals_for_dtw(ivt_by_pos[pos])

            if not native_sigs or not ivt_sigs:
                continue

            # Filter empty signals
            valid_native = [(n, s) for n, s in zip(native_names, native_sigs) if len(s) > 0]
            valid_ivt = [(n, s) for n, s in zip(ivt_names, ivt_sigs) if len(s) > 0]
            if not valid_native or not valid_ivt:
                continue
            native_names = [n for n, _ in valid_native]
            native_sigs = [s for _, s in valid_native]
            ivt_names = [n for n, _ in valid_ivt]
            ivt_sigs = [s for _, s in valid_ivt]

            all_sigs = native_sigs + ivt_sigs
            matrix = dtw_pairwise_varlen(
                all_sigs,
                use_open_start=False,
                use_open_end=False,
                use_cuda=None,
            )

            kmer = native_by_pos[pos].reference_kmer
            position_results[pos] = PositionResult(
                position=pos,
                reference_kmer=kmer,
                n_native_reads=len(native_names),
                n_ivt_reads=len(ivt_names),
                native_read_names=native_names,
                ivt_read_names=ivt_names,
                distance_matrix=matrix,
            )

        nat_depth = native_stats[contig].mean_depth if contig in native_stats else 0.0
        ivt_depth = ivt_stats[contig].mean_depth if contig in ivt_stats else 0.0
        results[contig] = ContigResult(
            contig=contig,
            native_depth=nat_depth,
            ivt_depth=ivt_depth,
            positions=position_results,
        )

    return results


def collect_read_level_predictions(
    hmm_results: dict,
    score_field: str = "p_mod_hmm",
) -> tuple[np.ndarray, np.ndarray]:
    """Collect (y_true, y_score) across all contigs."""
    y_true_all, y_score_all = [], []
    for contig, cmr in hmm_results.items():
        for pos, ps in cmr.position_stats.items():
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            scores = getattr(ps, score_field)
            if is_mod:
                y_true_pos = np.concatenate([np.ones(ps.n_native), np.zeros(ps.n_ivt)])
            else:
                y_true_pos = np.zeros(ps.n_native + ps.n_ivt)
            y_true_all.append(y_true_pos)
            y_score_all.append(scores)
    yt = np.concatenate(y_true_all)
    ys = np.concatenate(y_score_all)
    valid = ~np.isnan(ys)
    return yt[valid], ys[valid]


def compute_metrics(y_true: np.ndarray, y_score: np.ndarray) -> dict:
    """Compute AUROC, AUPRC, and F1 at optimal threshold."""
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):
        return {"auroc": np.nan, "auprc": np.nan, "f1": np.nan,
                "precision": np.nan, "recall": np.nan, "threshold": np.nan,
                "n_total": len(y_true), "n_pos": int(y_true.sum()),
                "n_neg": int(len(y_true) - y_true.sum())}  
    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    f1s = np.where(
        (precisions[:-1] + recalls[:-1]) > 0,
        2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
        0.0,
    )
    best_idx = np.argmax(f1s)
    return {
        "auroc": auroc, "auprc": auprc,
        "f1": float(f1s[best_idx]),
        "precision": float(precisions[:-1][best_idx]),
        "recall": float(recalls[:-1][best_idx]),
        "threshold": float(thresholds[best_idx]),
        "n_total": len(y_true), "n_pos": int(y_true.sum()),
        "n_neg": int(len(y_true) - y_true.sum()),
    }

In [ ]:
# Main benchmarking loop
bench_results = {}  # padding -> {contig_results, hmm_results, sites, metrics, timing}

for pad in PADDING_VALUES:
    print(f"\n{'='*70}")
    print(f"  PADDING = {pad}")
    print(f"{'='*70}")
    timings = {}

    # --- DTW computation ---
    gc.collect()
    mem_before = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    t0 = time.time()

    contig_results = compute_dtw_for_padding(pad, eventalign_signals)

    timings["dtw"] = time.time() - t0
    mem_after = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # ru_maxrss is in bytes on macOS, KB on Linux
    mem_delta_mb = (mem_after - mem_before) / (1024 * 1024) if sys.platform == "darwin" else (mem_after - mem_before) / 1024

    n_positions = sum(len(cr.positions) for cr in contig_results.values())
    avg_signal_len = []
    for cr in contig_results.values():
        for pr in cr.positions.values():
            avg_signal_len.append(pr.distance_matrix.shape[0])
    print(f"  DTW: {timings['dtw']:.1f}s, {n_positions} positions")
    print(f"  Memory delta: ~{mem_delta_mb:.1f} MB")

    # --- HMM pipeline ---
    t0 = time.time()

    hmm_results = {}
    for contig, cr in contig_results.items():
        hmm_results[contig] = compute_sequential_modification_probabilities(cr)

    timings["hmm"] = time.time() - t0
    print(f"  HMM pipeline: {timings['hmm']:.1f}s")

    # --- Site-level aggregation ---
    t0 = time.time()

    sites = aggregate_all(hmm_results)

    timings["aggregation"] = time.time() - t0
    print(f"  Aggregation: {timings['aggregation']:.1f}s, {len(sites)} sites")

    # --- Read-level metrics ---
    yt, ys = collect_read_level_predictions(hmm_results)
    metrics = compute_metrics(yt, ys)

    # --- Site-level metrics ---
    n_sig_005 = sum(1 for s in sites if s.padj < 0.05)
    n_sig_001 = sum(1 for s in sites if s.padj < 0.01)

    # True positive sites: known mod positions with padj < 0.05
    tp_sites = sum(
        1 for s in sites
        if s.padj < 0.05 and (s.contig, s.position) in KNOWN_MOD_PIPELINE_SET
    )
    # False positive sites: non-mod positions with padj < 0.05
    fp_sites = sum(
        1 for s in sites
        if s.padj < 0.05 and (s.contig, s.position) not in KNOWN_MOD_PIPELINE_SET
    )
    # Total known mod sites present in results
    total_known = sum(
        1 for s in sites
        if (s.contig, s.position) in KNOWN_MOD_PIPELINE_SET
    )

    site_precision = tp_sites / max(tp_sites + fp_sites, 1)
    site_recall = tp_sites / max(total_known, 1)

    timings["total"] = timings["dtw"] + timings["hmm"] + timings["aggregation"]

    print(f"\n  Read-level:  AUROC={metrics['auroc']:.4f}  AUPRC={metrics['auprc']:.4f}  F1={metrics['f1']:.4f}")
    print(f"  Site-level:  {n_sig_005} sig (padj<0.05), {n_sig_001} sig (padj<0.01)")
    print(f"  Site TP={tp_sites}, FP={fp_sites}, known={total_known}")
    print(f"  Site precision={site_precision:.4f}, recall={site_recall:.4f}")

    # Save site results TSV
    tsv_path = OUTPUT_DIR / f"site_results_padding{pad}.tsv"
    write_site_tsv(sites, tsv_path)
    print(f"  Saved: {tsv_path}")

    bench_results[pad] = {
        "contig_results": contig_results,
        "hmm_results": hmm_results,
        "sites": sites,
        "metrics": metrics,
        "yt": yt,
        "ys": ys,
        "timings": timings,
        "mem_delta_mb": mem_delta_mb,
        "n_positions": n_positions,
        "n_sig_005": n_sig_005,
        "n_sig_001": n_sig_001,
        "tp_sites": tp_sites,
        "fp_sites": fp_sites,
        "total_known": total_known,
        "site_precision": site_precision,
        "site_recall": site_recall,
    }

## 4. Comparison Table

In [ ]:
# Build comparison table
rows = []
for pad in PADDING_VALUES:
    b = bench_results[pad]
    m = b["metrics"]
    t = b["timings"]
    rows.append({
        "padding": pad,
        # Read-level
        "AUROC": m["auroc"],
        "AUPRC": m["auprc"],
        "F1": m["f1"],
        "Precision (read)": m["precision"],
        "Recall (read)": m["recall"],
        # Site-level
        "Sig sites (padj<0.05)": b["n_sig_005"],
        "Sig sites (padj<0.01)": b["n_sig_001"],
        "Site TP": b["tp_sites"],
        "Site FP": b["fp_sites"],
        "Site precision": b["site_precision"],
        "Site recall": b["site_recall"],
        # Resource usage
        "DTW time (s)": t["dtw"],
        "HMM time (s)": t["hmm"],
        "Agg time (s)": t["aggregation"],
        "Total time (s)": t["total"],
        "Mem delta (MB)": b["mem_delta_mb"],
        "Positions": b["n_positions"],
    })

df_comparison = pd.DataFrame(rows).set_index("padding")

print("Padding Benchmark Comparison")
print("=" * 90)
df_comparison.round(4)

## 5. ROC & PR Curves

In [ ]:
colors = {0: "#5B9BD5", 1: "#ED7D31", 2: "#70AD47"}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ROC
ax = axes[0]
for pad in PADDING_VALUES:
    b = bench_results[pad]
    yt, ys = b["yt"], b["ys"]
    if len(yt) > 0 and yt.sum() > 0:
        fpr, tpr, _ = roc_curve(yt, ys)
        auroc_val = roc_auc_score(yt, ys)
        ax.plot(fpr, tpr, color=colors[pad], linewidth=2.5,
                label=f"padding={pad} (AUROC={auroc_val:.4f})")

ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves by Padding", fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# PR
ax = axes[1]
for pad in PADDING_VALUES:
    b = bench_results[pad]
    yt, ys = b["yt"], b["ys"]
    if len(yt) > 0 and yt.sum() > 0:
        prec, rec, _ = precision_recall_curve(yt, ys)
        auprc_val = average_precision_score(yt, ys)
        ax.plot(rec, prec, color=colors[pad], linewidth=2.5,
                label=f"padding={pad} (AUPRC={auprc_val:.4f})")

prevalence = bench_results[0]["metrics"]["n_pos"] / max(bench_results[0]["metrics"]["n_total"], 1)
ax.axhline(prevalence, color="gray", linestyle="--", linewidth=1, label=f"baseline ({prevalence:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves by Padding", fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_by_padding.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Site-Level P-value Distributions

In [ ]:
fig, axes = plt.subplots(1, len(PADDING_VALUES), figsize=(6 * len(PADDING_VALUES), 5))
if len(PADDING_VALUES) == 1:
    axes = [axes]

for ax, pad in zip(axes, PADDING_VALUES):
    sites = bench_results[pad]["sites"]
    pvals_mod = [-np.log10(s.pvalue + 1e-300) for s in sites if (s.contig, s.position) in KNOWN_MOD_PIPELINE_SET]
    pvals_unmod = [-np.log10(s.pvalue + 1e-300) for s in sites if (s.contig, s.position) not in KNOWN_MOD_PIPELINE_SET]

    bins = np.linspace(0, max(max(pvals_mod, default=1), max(pvals_unmod, default=1)) + 1, 40)
    ax.hist(pvals_unmod, bins=bins, alpha=0.6, color="#5B9BD5", label=f"Unmodified (n={len(pvals_unmod)})")
    ax.hist(pvals_mod, bins=bins, alpha=0.7, color="#ED7D31", label=f"Modified (n={len(pvals_mod)})")
    ax.axvline(-np.log10(0.05), color="red", linestyle="--", linewidth=1, label="p=0.05")
    ax.set_xlabel("-log10(p-value)")
    ax.set_ylabel("Count")
    ax.set_title(f"padding={pad}", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("P-value Distributions: Modified vs Unmodified Sites", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pvalue_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Resource Usage Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pads = PADDING_VALUES
bar_colors = [colors[p] for p in pads]
x = np.arange(len(pads))
width = 0.5

# Time breakdown (stacked bar)
ax = axes[0]
dtw_times = [bench_results[p]["timings"]["dtw"] for p in pads]
hmm_times = [bench_results[p]["timings"]["hmm"] for p in pads]
agg_times = [bench_results[p]["timings"]["aggregation"] for p in pads]

ax.bar(x, dtw_times, width, label="DTW", color="#5B9BD5")
ax.bar(x, hmm_times, width, bottom=dtw_times, label="HMM", color="#ED7D31")
ax.bar(x, agg_times, width, bottom=[d + h for d, h in zip(dtw_times, hmm_times)],
       label="Aggregation", color="#70AD47")
ax.set_xticks(x)
ax.set_xticklabels([f"pad={p}" for p in pads])
ax.set_ylabel("Time (seconds)")
ax.set_title("Wall Time Breakdown", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Accuracy metrics grouped bar
ax = axes[1]
aurocs = [bench_results[p]["metrics"]["auroc"] for p in pads]
auprcs = [bench_results[p]["metrics"]["auprc"] for p in pads]
f1s = [bench_results[p]["metrics"]["f1"] for p in pads]

w = 0.25
ax.bar(x - w, aurocs, w, label="AUROC", color="#5B9BD5")
ax.bar(x, auprcs, w, label="AUPRC", color="#ED7D31")
ax.bar(x + w, f1s, w, label="F1", color="#70AD47")
ax.set_xticks(x)
ax.set_xticklabels([f"pad={p}" for p in pads])
ax.set_ylabel("Score")
ax.set_title("Read-Level Accuracy", fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")

# Site-level TP/FP
ax = axes[2]
tps = [bench_results[p]["tp_sites"] for p in pads]
fps = [bench_results[p]["fp_sites"] for p in pads]

ax.bar(x - 0.15, tps, 0.3, label="True Positives", color="#70AD47")
ax.bar(x + 0.15, fps, 0.3, label="False Positives", color="#C00000")
ax.set_xticks(x)
ax.set_xticklabels([f"pad={p}" for p in pads])
ax.set_ylabel("Number of Sites")
ax.set_title("Site-Level Calls (padj < 0.05)", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "resource_usage.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Site-Level mod_ratio at Known Modification Sites

In [ ]:
# Compare mod_ratio estimates at known modification sites across padding values
mod_site_rows = []
for pad in PADDING_VALUES:
    for s in bench_results[pad]["sites"]:
        if (s.contig, s.position) in KNOWN_MOD_PIPELINE_SET:
            mod_short = KNOWN_MOD_PIPELINE[(s.contig, s.position)][0]
            mod_site_rows.append({
                "padding": pad,
                "contig": s.contig,
                "position": s.position,
                "bio_position": s.position + POSITION_OFFSET,
                "mod_type": mod_short,
                "mod_ratio": s.mod_ratio,
                "ci_low": s.ci_low,
                "ci_high": s.ci_high,
                "pvalue": s.pvalue,
                "padj": s.padj,
                "effect_size": s.effect_size,
                "n_native": s.n_native,
            })

df_mod_sites = pd.DataFrame(mod_site_rows)

if len(df_mod_sites) > 0:
    # Pivot: one row per site, columns per padding
    pivot = df_mod_sites.pivot_table(
        index=["contig", "position", "bio_position", "mod_type"],
        columns="padding",
        values=["mod_ratio", "padj", "effect_size"],
    ).round(4)
    print("Known Modification Sites: mod_ratio and padj by padding")
    print("=" * 100)
    pivot
else:
    print("No known modification sites found in results.")

In [ ]:
# Scatter: mod_ratio at padding=0 vs padding=1 vs padding=2
if len(df_mod_sites) > 0 and len(PADDING_VALUES) >= 2:
    fig, axes = plt.subplots(1, len(PADDING_VALUES) - 1, figsize=(7 * (len(PADDING_VALUES) - 1), 6))
    if len(PADDING_VALUES) == 2:
        axes = [axes]

    for ax_idx, (pad_a, pad_b) in enumerate(zip(PADDING_VALUES[:-1], PADDING_VALUES[1:])):
        ax = axes[ax_idx]
        df_a = df_mod_sites[df_mod_sites["padding"] == pad_a].set_index(["contig", "position"])
        df_b = df_mod_sites[df_mod_sites["padding"] == pad_b].set_index(["contig", "position"])
        common_idx = df_a.index.intersection(df_b.index)

        if len(common_idx) > 0:
            x_vals = df_a.loc[common_idx, "mod_ratio"].values
            y_vals = df_b.loc[common_idx, "mod_ratio"].values
            ax.scatter(x_vals, y_vals, alpha=0.7, edgecolor="black", linewidth=0.3, s=40)
            ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
            ax.set_xlabel(f"mod_ratio (padding={pad_a})")
            ax.set_ylabel(f"mod_ratio (padding={pad_b})")
            ax.set_title(f"mod_ratio: pad={pad_a} vs pad={pad_b}", fontweight="bold")
            ax.set_xlim(-0.05, 1.05)
            ax.set_ylim(-0.05, 1.05)
            ax.grid(True, alpha=0.3)
            # Correlation
            corr = np.corrcoef(x_vals, y_vals)[0, 1]
            ax.text(0.05, 0.92, f"r = {corr:.3f}", transform=ax.transAxes, fontsize=11)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "mod_ratio_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Summary

In [ ]:
print("=" * 90)
print("PADDING BENCHMARK SUMMARY")
print("=" * 90)
print(f"\nDataset: E. coli rRNA ({len(eventalign_signals)} contigs)")
print(f"Known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"f5c time (shared): {f5c_time:.1f}s")

col_w = 14
headers = [f"pad={p}" for p in PADDING_VALUES]
print(f"\n{'Metric':<28s}" + "".join(f" {h:>{col_w}s}" for h in headers))
print("-" * (28 + (col_w + 1) * len(headers)))

metric_rows = [
    ("Read-level AUROC", lambda p: bench_results[p]["metrics"]["auroc"], ".4f"),
    ("Read-level AUPRC", lambda p: bench_results[p]["metrics"]["auprc"], ".4f"),
    ("Read-level F1", lambda p: bench_results[p]["metrics"]["f1"], ".4f"),
    ("Site sig (padj<0.05)", lambda p: bench_results[p]["n_sig_005"], "d"),
    ("Site sig (padj<0.01)", lambda p: bench_results[p]["n_sig_001"], "d"),
    ("Site TP", lambda p: bench_results[p]["tp_sites"], "d"),
    ("Site FP", lambda p: bench_results[p]["fp_sites"], "d"),
    ("Site precision", lambda p: bench_results[p]["site_precision"], ".4f"),
    ("Site recall", lambda p: bench_results[p]["site_recall"], ".4f"),
    ("DTW time (s)", lambda p: bench_results[p]["timings"]["dtw"], ".1f"),
    ("HMM time (s)", lambda p: bench_results[p]["timings"]["hmm"], ".1f"),
    ("Total time (s)", lambda p: bench_results[p]["timings"]["total"], ".1f"),
    ("Memory delta (MB)", lambda p: bench_results[p]["mem_delta_mb"], ".1f"),
    ("Positions", lambda p: bench_results[p]["n_positions"], "d"),
]

for label, fn, fmt in metric_rows:
    vals = [fn(p) for p in PADDING_VALUES]
    print(f"  {label:<26s}" + "".join(f" {v:>{col_w}{fmt}}" for v in vals))

# Time breakdown
print(f"\nTime breakdown (DTW dominates at higher padding):")
for pad in PADDING_VALUES:
    t = bench_results[pad]["timings"]
    total = t["total"]
    print(f"  pad={pad}: DTW {t['dtw']:.1f}s ({100*t['dtw']/total:.0f}%) | "
          f"HMM {t['hmm']:.1f}s ({100*t['hmm']/total:.0f}%) | "
          f"Agg {t['aggregation']:.1f}s ({100*t['aggregation']/total:.0f}%)")

print(f"\n" + "=" * 90)

In [ ]:
# Save comparison table
df_comparison.to_csv(OUTPUT_DIR / "padding_comparison.csv")
print(f"Comparison table saved to: {OUTPUT_DIR / 'padding_comparison.csv'}")

if len(df_mod_sites) > 0:
    df_mod_sites.to_csv(OUTPUT_DIR / "known_mod_sites_by_padding.csv", index=False)
    print(f"Known mod sites saved to: {OUTPUT_DIR / 'known_mod_sites_by_padding.csv'}")

print(f"\nSite-level TSVs saved:")
for pad in PADDING_VALUES:
    print(f"  padding={pad}: {OUTPUT_DIR / f'site_results_padding{pad}.tsv'}")

---

**Done.** Outputs saved to `output/padding_benchmark/`:

- `site_results_padding{0,1,2}.tsv` — site-level results per padding
- `padding_comparison.csv` — full metrics table
- `known_mod_sites_by_padding.csv` — mod_ratio at known sites per padding
- PNG plots: ROC/PR curves, p-value distributions, resource usage, mod_ratio scatter